
# Özellik Çıkarımı (Feature Engineering) ve Zamansal Veri Ayrımı
Bu not defteri, veri setine yeni, anlamlı öznitelikler (features) eklemeyi ve hedef sızıntısını (target leakage) engelleyerek veriyi zaman ekseninde bölmeyi (temporal split) amaçlar.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Src modüllerini yola ekleme
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.data.load_data import load_raw_data
from src.features.build_features import build_all_features
from src.data.split_data import temporal_train_val_test_split



## Verinin Yüklenmesi ve Özellik Çıkarımı
Büyük ham veri yüklenir ve `src.features.build_features` içerisindeki fonksiyon aracılığıyla yeni öznitelikler türetilir.


In [ ]:

DATA_PATH = '../data/raw/PS_20174392719_1491204439457_log.csv'
try:
    df_raw = load_raw_data(DATA_PATH)
    print("Ham veri yüklendi. Satır sayısı:", len(df_raw))
    
    # Feature Engineering uygula
    df_features = build_all_features(df_raw)
    print("Özellik çıkarımı tamamlandı. Yeni sütun sayısı:", len(df_features.columns))
except Exception as e:
    print(f"Hata: {e}")



## Yeni Özelliklerin İncelenmesi
Oluşturduğumuz bazı kritik özellikleri (örneğin bakiye hatalarını) fraud olan ve olmayan işlemler için kıyaslayalım.


In [ ]:

if 'df_features' in locals():
    # Görselleştirme için örneklem alalım
    df_sample = df_features.sample(n=min(len(df_features), 300000), random_state=42)
    
    plt.figure(figsize=(10,6))
    # Yalnızca TRANSFER ve CASH_OUT işlemlerine bakalım (sadece buralarda fraud var)
    mask = df_sample['is_risky_type'] == 1
    
    sns.boxplot(data=df_sample[mask], x='isFraud', y='abs_error_balance_orig')
    plt.yscale('symlog')
    plt.title('Kaynak Hesap Bakiye Hatası Mutlak Değeri (is_risky_type=1 olanlar)')
    plt.ylabel('Mutlak Bakiye Hatası (SymLog)')
    plt.show()



Yukarıdaki grafikte fraud işlemlerinin çok yüksek bakiye hataları taşıdığı (sistemin parayı çekmiş gibi göstermemesi vb.) çok net görülebilmektedir. Bu durum, simülasyon verisinin ürettiği çok güçlü bir sinyaldir (neredeyse leakage seviyesinde bir bulgudur). Bu nedenle `error_balance_orig` karar ağaçları için en kritik değişken olacaktır.



## Leakage-Safe Zamansal Veri Ayrımı (Temporal Split)
Finansal işlemlerde geçmiş verilere bakarak geleceği tahmin ederiz. Bu yüzden veriyi rastgele `train_test_split` ile değil, zamansal `step` değişkenine göre sıralayarak bölmeliyiz.
- Train: %70 (En eski veriler)
- Validation: %15 (Ortadaki veriler)
- Test: %15 (En yeni veriler)


In [ ]:

if 'df_features' in locals():
    train_df, val_df, test_df = temporal_train_val_test_split(df_features, train_size=0.70, val_size=0.15)
    
    print(f"Train Seti: {len(train_df)} satır. Step aralığı: {train_df['step'].min()} - {train_df['step'].max()}")
    print(f"Val Seti: {len(val_df)} satır. Step aralığı: {val_df['step'].min()} - {val_df['step'].max()}")
    print(f"Test Seti: {len(test_df)} satır. Step aralığı: {test_df['step'].min()} - {test_df['step'].max()}")
    
    # Kesişim kontrolü
    assert set(train_df['step']).isdisjoint(set(val_df['step'])), "Train ve Val setleri aynı step değerini paylaşıyor! Leakage riski."
    assert set(val_df['step']).isdisjoint(set(test_df['step'])), "Val ve Test setleri aynı step değerini paylaşıyor! Leakage riski."
    print("Sızıntı (Leakage) kontrolleri başarıyla geçildi.")



## Modelleme Aşamasında Dışlanacak Değişkenler
Eğitim aşamasında aşağıdaki değişkenler X (features) matrisinden çıkartılmalıdır:
1. `isFraud` (Hedef değişken)
2. `isFlaggedFraud` (Mevcut kural sistemi, model ile kıyaslamak için tutulur ama modele verilmez)
3. (Daha önce çıkartılmış olan) `nameOrig`, `nameDest` yüksek kardinaliteli kimlik bilgileri.
